# 61 — Region-Constrained Experimental Design

**Core statement**

Experiments validate admissible regions before selecting operating points.

This notebook turns region-constrained design theory into a laboratory workflow: simulate the admissible region, measure it, validate connected topology, compute distance to failure, and operate inside the robust interior.

## Runtime requirements

```bash
pip install numpy pandas matplotlib scipy ipython
```

In [ ]:
from pathlib import Path
import json
import shutil
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle, FancyBboxPatch, Circle, Ellipse
from scipy import ndimage
from IPython.display import FileLink, display

OUT = Path("outputs/notebook_61_region_constrained_experimental_design")
FIG = OUT / "figures"
DATA = OUT / "data"
FIG.mkdir(parents=True, exist_ok=True)
DATA.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({
    "figure.dpi": 140,
    "savefig.dpi": 180,
    "font.size": 11,
    "axes.titlesize": 18,
    "axes.labelsize": 12,
})

def savefig(fig, name):
    path = FIG / name
    fig.savefig(path, bbox_inches="tight")
    return path

def clean_axes(ax):
    ax.set_xticks([])
    ax.set_yticks([])
    for spine in ax.spines.values():
        spine.set_visible(False)

def box(ax, x, y, w, h, title, subtitle="", lw=2.0, rounded=True, title_size=11, subtitle_size=8):
    patch = FancyBboxPatch((x,y), w,h, boxstyle="round,pad=0.02,rounding_size=0.025",
                           fill=False, linewidth=lw, edgecolor="black") if rounded else Rectangle((x,y), w,h, fill=False, linewidth=lw, edgecolor="black")
    ax.add_patch(patch)
    ax.text(x+w/2, y+h*0.62, title, ha="center", va="center", fontsize=title_size, fontweight="bold")
    if subtitle:
        ax.text(x+w/2, y+h*0.27, subtitle, ha="center", va="center", fontsize=subtitle_size)
    return patch

def arrow(ax, start, end, lw=2.0):
    ax.annotate("", xy=end, xytext=start,
                arrowprops=dict(arrowstyle="->", linewidth=lw, shrinkA=0, shrinkB=0))

def chain_horizontal(ax, items, y=0.50, x0=0.05, x1=0.95, w=None, h=0.16, title_size=10, subtitle_size=8):
    n = len(items)
    if w is None:
        w = min(0.12, (x1-x0)/(n*1.25))
    xs = np.linspace(x0, x1-w, n)
    for i,(t,s) in enumerate(items):
        box(ax, xs[i], y, w, h, t, s, title_size=title_size, subtitle_size=subtitle_size)
        if i < n-1:
            arrow(ax, (xs[i]+w+0.01, y+h/2), (xs[i+1]-0.01, y+h/2), lw=2)
    return xs

def chain_vertical(ax, items, x=0.5, y0=0.78, dy=0.12, w=0.42, h=0.075, title_size=11):
    for i,(t,s) in enumerate(items):
        y = y0 - i*dy
        box(ax, x-w/2, y, w, h, t, s, title_size=title_size)
        if i < len(items)-1:
            arrow(ax, (x, y), (x, y-dy+h), lw=1.8)

N = 280
x = np.linspace(0, 1, N)
y = np.linspace(0, 1, N)
X, Y = np.meshgrid(x, y)
threshold = 0.38

def sigmoid(z, s=0.045):
    return 1/(1+np.exp(-z/s))

def region_field(center=0.62, width=0.25, yscale=0.22, amp=1.0, bump=0.25, noise=0.0, shift=(0,0), split=False):
    Xs = X - shift[0]
    Ys = Y - shift[1]
    plateau = sigmoid(Xs-(center-width), 0.045) * sigmoid((center+width)-Xs, 0.045)
    low_y = np.exp(-(Ys/yscale)**1.55)
    shoulder = bump*np.exp(-((Xs-(center+0.18))**2/0.018 + (Ys-0.08)**2/0.025))
    Z = amp*np.clip(plateau*low_y + shoulder, 0, 1)
    if split:
        Z -= 0.65*np.exp(-((Xs-center)**2/0.004 + (Ys-0.09)**2/0.04))
    if noise:
        rng = np.random.default_rng(61)
        Z += noise*rng.normal(0,1,Z.shape)
        Z = ndimage.gaussian_filter(Z, 1.1)
    return np.clip(Z,0,1)

def metrics(Z, threshold=threshold):
    mask = Z >= threshold
    labels, ncomp = ndimage.label(mask)
    if ncomp == 0:
        return dict(mask=mask, largest=np.zeros_like(mask), dist=np.zeros_like(Z), ncomp=0,
                    area=0.0, largest_area=0.0, max_margin=0.0, avg_margin=0.0, xstar=np.nan, ystar=np.nan)
    sizes = ndimage.sum(mask, labels, index=np.arange(1, ncomp+1))
    lid = int(np.argmax(sizes)+1)
    largest = labels == lid
    dist_px = ndimage.distance_transform_edt(largest)
    dist = dist_px / dist_px.max() if dist_px.max() else dist_px
    idx = np.unravel_index(np.argmax(dist_px), dist_px.shape)
    return dict(mask=mask, largest=largest, dist=dist, ncomp=int(ncomp),
                area=float(mask.mean()), largest_area=float(largest.mean()),
                max_margin=float(dist_px.max()/N), avg_margin=float(dist[largest].mean()) if largest.any() else 0.0,
                xstar=float(x[idx[1]]), ystar=float(y[idx[0]]))

def plot_region(ax, Z, title="", show_dist=False, annotate=False):
    M = metrics(Z)
    image = M["dist"] if show_dist else Z
    if show_dist:
        image = np.where(M["largest"], image, np.nan)
    im = ax.imshow(image, origin="lower", extent=[0,1,0,1], aspect="auto", vmin=0, vmax=1)
    ax.contour(X,Y,Z,levels=[threshold],colors="black",linewidths=1.8)
    ax.contour(X,Y,M["largest"].astype(float),levels=[0.5],colors="black",linewidths=2.6)
    ax.scatter([M["xstar"]],[M["ystar"]],s=80,zorder=4)
    ax.set_title(title)
    ax.set_xlabel("control 1")
    return im, M

## 00 Context

Experiments validate regions before optimizing points.

In [ ]:
fig, ax = plt.subplots(figsize=(13,4.8))
clean_axes(ax); ax.set_xlim(0,1); ax.set_ylim(0,1)
ax.set_title("Region-Constrained Experimental Design")
items=[
    ("Physics","resources"),
    ("Simulation","predict Ω"),
    ("Experiment","measure Ω"),
    ("Topology","validate Ωc"),
    ("Robust Operation","choose x*"),
    ("Computation","execute"),
]
chain_horizontal(ax, items, y=0.48, w=0.135, h=0.19, title_size=9, subtitle_size=7)
ax.text(0.5,0.19,"Experiments validate admissible regions before selecting operating points.",ha="center",fontsize=14)
savefig(fig,"61_00_region_constrained_experimental_design.png")
plt.show()

## 07 Simulation vs Experiment

The validated region is the overlap between simulated admissibility and measured admissibility.

In [ ]:
Zsim = region_field(center=0.61,width=0.30,yscale=0.24,bump=0.22)
Zexp = region_field(center=0.65,width=0.24,yscale=0.21,bump=0.18,noise=0.02,shift=(0.015,-0.005))
Msim, Mexp = metrics(Zsim), metrics(Zexp)
intersection = np.logical_and(Msim["mask"], Mexp["mask"]).astype(float)

fig, axes = plt.subplots(1,3,figsize=(13,4.2),sharex=True,sharey=True)
ims=[]
for ax, Z, title in zip(axes[:2],[Zsim,Zexp],["simulation predicts Ω", "experiment measures Ω"]):
    im,_ = plot_region(ax,Z,title)
    ims.append(im)
axes[2].imshow(intersection, origin="lower", extent=[0,1,0,1], aspect="auto", vmin=0, vmax=1)
axes[2].contour(X,Y,intersection,levels=[0.5],colors="black",linewidths=2.4)
axes[2].set_title("validated region Ωsim ∩ Ωexp")
axes[2].set_xlabel("control 1")
axes[0].set_ylabel("control 2")
fig.suptitle("Simulation and Experiment Meet in the Region",fontsize=18)
fig.text(0.5,0.02,"Good theories preserve connected admissible regions under measurement.",ha="center",fontsize=12)
savefig(fig,"61_07_simulation_experiment_intersection.png")
plt.show()

## 17 Experimental Refinement

Experimental progress means expanding and connecting Ωc before choosing x*.

In [ ]:
fields = {
    "iteration 1\nsmall Ω": region_field(center=0.58,width=0.15,yscale=0.15,bump=0.10,split=True),
    "iteration 2\nbetter calibration": region_field(center=0.60,width=0.20,yscale=0.18,bump=0.14),
    "iteration 3\nconnected Ωc": region_field(center=0.62,width=0.25,yscale=0.22,bump=0.18),
    "iteration 4\nrobust interior": region_field(center=0.64,width=0.33,yscale=0.29,bump=0.25),
}
records=[]
fig, axes = plt.subplots(1,4,figsize=(14,3.9),sharex=True,sharey=True)
for ax,(name,Zi) in zip(axes,fields.items()):
    im,M = plot_region(ax,Zi,name)
    ax.text(0.05,0.86,f"|Ωc|={M['largest_area']:.2f}\nM={M['max_margin']:.2f}",transform=ax.transAxes,fontsize=9,fontweight="bold")
    records.append({"stage":name.replace("\n"," "), "largest_component":M["largest_area"], "max_margin":M["max_margin"], "components":M["ncomp"]})
axes[0].set_ylabel("loss")
fig.suptitle("Experimental Refinement: Discover the Region Before Choosing the Point",fontsize=18)
pd.DataFrame(records).to_csv(DATA/"61_experimental_refinement_metrics.csv",index=False)
savefig(fig,"61_17_experimental_refinement.png")
plt.show()

## 23 Calibration Preserves Topology

Calibration is valuable because it preserves the connected admissible region and widens the distance field.

In [ ]:
uncal = region_field(center=0.58,width=0.18,yscale=0.16,bump=0.11,split=True,noise=0.01)
cal = region_field(center=0.62,width=0.29,yscale=0.25,bump=0.22,noise=0.005)
fig, axes = plt.subplots(1,4,figsize=(14,4),sharex=True,sharey=True)
for ax,Z,title,dist in [
    (axes[0],uncal,"uncalibrated Ω",False),
    (axes[1],uncal,"uncalibrated distance",True),
    (axes[2],cal,"calibrated Ω",False),
    (axes[3],cal,"calibrated distance",True),
]:
    im,M=plot_region(ax,Z,title,show_dist=dist)
    ax.text(0.05,0.86,f"|Ωc|={M['largest_area']:.2f}\nM={M['max_margin']:.2f}",transform=ax.transAxes,fontsize=9,fontweight="bold")
axes[0].set_ylabel("loss")
fig.suptitle("Calibration Preserves Topology Before It Improves the Point",fontsize=18)
savefig(fig,"61_23_calibration_preserves_topology.png")
plt.show()

## 31 Region-First Experimentation

Point-first experimentation learns locally; region-first experimentation learns the admissible set.

In [ ]:
fig, axes = plt.subplots(1,2,figsize=(12,6))
for ax in axes:
    clean_axes(ax); ax.set_xlim(0,1); ax.set_ylim(0,1)
axes[0].set_title("Point-first experiment")
chain_vertical(axes[0],[("guess parameter","x"),("run","single trial"),("measure","local score"),("repeat","adjust point")],x=0.5,y0=0.68,dy=0.16,w=0.50,h=0.08)
axes[0].text(0.5,0.15,"learns one point at a time",ha="center",fontsize=12)

axes[1].set_title("Region-constrained experiment")
chain_vertical(axes[1],[("simulate","candidate Ω"),("measure","experimental Ω"),("validate topology","Ωc"),("compute margin","d(∂Ωc)"),("operate","x*")],x=0.5,y0=0.74,dy=0.13,w=0.52,h=0.075)
axes[1].text(0.5,0.15,"learns the admissible region first",ha="center",fontsize=12)
fig.suptitle("Region-First Experimentation",fontsize=19)
savefig(fig,"61_31_region_first_experimentation.png")
plt.show()

## 41 Why Experiments Fail

Failure follows loss of admissible topology or margin, not merely bad optimization.

In [ ]:
fig, ax = plt.subplots(figsize=(12,5.2))
clean_axes(ax); ax.set_xlim(0,1); ax.set_ylim(0,1)
ax.set_title("Why Experiments Fail")
causes=[
    ("bad resources","physics cannot support Ω"),
    ("wrong constraints","Ω is misidentified"),
    ("broken topology","Ωc disconnects"),
    ("boundary operation","x* is too close to failure"),
]
xs=np.linspace(0.08,0.74,4)
for x0,(t,s) in zip(xs,causes):
    box(ax,x0,0.50,0.18,0.18,t,s,lw=2.2,title_size=10,subtitle_size=7)
    arrow(ax,(x0+0.09,0.50),(0.50,0.30),lw=1.6)
box(ax,0.35,0.18,0.30,0.10,"failed operation","not merely bad optimization",lw=2.4,title_size=11,subtitle_size=8)
ax.text(0.5,0.08,"Failure follows loss of admissible topology or margin.",ha="center",fontsize=13)
savefig(fig,"61_41_why_experiments_fail.png")
plt.show()

## 47 Universal Laboratory Workflow

A strong experimental claim preserves the region, not only the best point.

In [ ]:
fig, ax = plt.subplots(figsize=(14,4.8))
clean_axes(ax); ax.set_xlim(0,1); ax.set_ylim(0,1)
ax.set_title("Universal Laboratory Workflow")
items=[
    ("Physics","resources"),
    ("Simulation","model"),
    ("Measurement","data"),
    ("Ω","admissible region"),
    ("Ωc","connected family"),
    ("Distance","margin"),
    ("Operation","x*"),
    ("Paper","claim"),
]
chain_horizontal(ax, items, y=0.48, w=0.105, h=0.17, title_size=9, subtitle_size=7)
ax.text(0.5,0.18,"A strong experimental claim preserves the region, not only the best point.",ha="center",fontsize=13)
savefig(fig,"61_47_universal_laboratory_workflow.png")
plt.show()

## 53 Cross-Disciplinary Experimental Objects

In [ ]:
apps = pd.DataFrame([
    ["Quantum hardware","admissible controls","connected control family","operating pulse"],
    ["Photonics","fabrication window","stable device family","device setting"],
    ["Atom interferometry","allowed trajectories","common-mode stable path","measurement protocol"],
    ["Machine learning","feasible models","robust basin","trained network"],
    ["Robotics","reachable states","safe trajectory family","controller"],
    ["Climate modeling","admissible scenarios","connected policy/physics region","robust intervention"],
], columns=["Field","Ω","Ωc","x*"])
apps.to_csv(DATA/"61_cross_domain_experimental_design.csv",index=False)

fig, ax = plt.subplots(figsize=(12,4.5))
clean_axes(ax)
ax.set_title("Cross-Disciplinary Experimental Objects")
tbl=ax.table(cellText=apps.values,colLabels=apps.columns,cellLoc="center",loc="center")
tbl.auto_set_font_size(False); tbl.set_fontsize(8.5); tbl.scale(1,1.55)
for (r,c),cell in tbl.get_celld().items():
    cell.set_linewidth(1)
    if r==0: cell.set_text_props(fontweight="bold")
savefig(fig,"61_53_cross_domain_objects.png")
plt.show()
apps

## 59 Conclusion

In [ ]:
fig, ax = plt.subplots(figsize=(12,6.2))
clean_axes(ax); ax.set_xlim(0,1); ax.set_ylim(0,1)
ax.set_title("Conclusion")
ax.text(0.5,0.72,"Experiments should discover regions",ha="center",fontsize=24,fontweight="bold")
ax.text(0.5,0.57,"before optimizing points",ha="center",fontsize=22)
items=[("Physics",""),("Topology",""),("Robustness",""),("Operation",""),("Discovery","")]
chain_horizontal(ax,items,y=0.24,x0=0.18,x1=0.82,w=0.12,h=0.10,title_size=10)
ax.text(0.5,0.10,"The experimental object is the admissible connected region; the operating point is one realization inside it.",ha="center",fontsize=12)
savefig(fig,"61_59_conclusion_experiments_discover_regions.png")
plt.show()

## 67 Experimental Evidence Ladder

In [ ]:
fig, ax = plt.subplots(figsize=(10.5,5.5))
clean_axes(ax); ax.set_xlim(0,1); ax.set_ylim(0,1)
ax.set_title("Experimental Evidence Ladder")
steps=[
    ("existence","can the resource be observed?"),
    ("admissibility","does it survive constraints?"),
    ("connectivity","does Ωc remain navigable?"),
    ("robustness","does margin survive perturbation?"),
    ("operation","does x* execute repeatedly?"),
]
chain_vertical(ax,steps,x=0.5,y0=0.76,dy=0.13,w=0.58,h=0.075,title_size=11)
ax.text(0.5,0.08,"Evidence strengthens when each stage preserves the previous specification.",ha="center",fontsize=12)
savefig(fig,"61_67_experimental_evidence_ladder.png")
plt.show()

## 71 Export summary metadata

In [ ]:
summary = {
    "notebook": "61_region_constrained_experimental_design",
    "thesis": "Experiments validate admissible regions before selecting operating points.",
    "pipeline": ["Physics", "Simulation", "Experiment", "Topology", "Robust Operation", "Computation"],
    "principles": [
        "Simulation predicts Ω.",
        "Experiment measures Ω.",
        "The validated region is Ω_sim ∩ Ω_exp.",
        "Calibration preserves Ωc before it improves the operating point.",
        "Robust operation selects x* from the interior of Ωc."
    ],
    "figure_count": len(list(FIG.glob("*.png"))),
    "data_files": sorted([p.name for p in DATA.glob("*")]),
}
(DATA/"61_experimental_design_summary.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")
summary

## 79 Package and download outputs

In [ ]:
# ============================================================
# Package and download Notebook 61 outputs
# ============================================================
#
# Run this final cell after executing the notebook.
#
# It creates:
#   61_region_constrained_experimental_design_outputs.zip
#
# containing:
#   figures/*.png
#   data/*.csv
#   data/*.json
#   README_61_outputs.md
#
# In Google Colab, it triggers an automatic browser download.
# In Jupyter, it displays a clickable download link.

package_root = Path("outputs/notebook_61_region_constrained_experimental_design")
figure_dir = package_root / "figures"
data_dir = package_root / "data"

figure_files = sorted(figure_dir.glob("*.png"))
data_files = sorted([p for p in data_dir.glob("*") if p.is_file()])

manifest = package_root / "README_61_outputs.md"
manifest.write_text(
    "# Notebook 61 Outputs\n\n"
    "Generated from `61_region_constrained_experimental_design.ipynb`.\n\n"
    "## Figures\n"
    + ("\n".join(f"- figures/{p.name}" for p in figure_files) if figure_files else "- No figures found. Run all cells before packaging.")
    + "\n\n## Data\n"
    + ("\n".join(f"- data/{p.name}" for p in data_files) if data_files else "- No data files found. Run all cells before packaging.")
    + "\n",
    encoding="utf-8",
)

archive_path = Path(shutil.make_archive("61_region_constrained_experimental_design_outputs", "zip", root_dir=package_root)).resolve()

print(f"Packaged outputs: {archive_path}")
print(f"Figures: {len(figure_files)}")
print(f"Data files: {len(data_files)}")

try:
    from google.colab import files
    files.download(str(archive_path))
except Exception:
    display(FileLink(str(archive_path)))